# Cell Segmentation with DeepCell

This notebook performs cell and nuclear segmentation on OME-TIFF images using DeepCell Mesmer. The pipeline is structured as follows:

1. **Load and prepare images** - Extract channels and optionally apply tissue masks
2. **Run DeepCell segmentation** - Segment nuclei and/or whole cells
3. **Extract cell features** - Morphological and intensity features for each cell and nucleus
4. **Save outputs** - Features CSV, segmentation masks, and cell/nuclear boundaries as GeoJSON

This notebook processes ROI images extracted from Step2 and prepares data for downstream phenotyping analysis.

**Summary:** This notebook performs cell segmentation using the **DeepCell** model. It processes extracted ROI (Region of Interest) images from Step2, identifies cells based on the DAPI channel, and extracts detailed morphological and intensity features for each cell.

**Outputs:**
- **Cell Features (CSV):** `outputs/segmentation/deepcell/features/`
- **Segmentation Masks (TIFF):** `outputs/segmentation/deepcell/masks/`
- **Cell Boundaries (GeoJSON):** `outputs/segmentation/deepcell/boundaries/`

These outputs are designed for downstream analysis, such as cell phenotyping and spatial analysis.

## Setup and Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Add src directory to Python path
project_root = Path('..').resolve()
src_path = project_root / 'src'
sys.path.insert(0, str(src_path))

# Import our modules
from spacemap.config import setup_logging, log_step
from spacemap.segmentation_deepcell import (
    DeepCellConfig,
    segment_image_deepcell,
    batch_segment_images,
    get_channel_names,
    check_segmentation_outputs_exist
)

# Setup logging
logger = setup_logging('segmentation_deepcell')
print(" Setup completed successfully!")
print(f" Project root: {project_root}")
print(f" Source path: {src_path}")

## Configuration Parameters

Configure the DeepCell segmentation parameters below. These can be adjusted based on your specific images and requirements.

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Input/Output paths
INPUT_DIR = "../outputs/registration/roi/"  # Directory with ROI images from Step2
OUTPUT_DIR = "../outputs/segmentation/deepcell/"  # Output for DeepCell results

# File pattern to match
FILE_PATTERN = "*.ome.tiff"  # or "*.ome.tif", adjust as needed

# Channel configuration
DAPI_CHANNEL = 'DAPI'  # Name of DAPI/nuclear channel

# Markers to use for segmentation (None = use all non-DAPI channels)
SEGMENTATION_MARKERS = [
    'Pan-Cytokeratin', 'E-cadherin', 'Vimentin', 'CD45', 'CD44',
    'aSMA', 'CD45RO', 'CD4', 'CD8', 'CD3e', 'CD31', 'CD20',
    'CD11c', 'CD69', 'CD11b', 'HLA-A', 'CD163', 'CD38'
]

# Create DeepCell segmentation configuration
config = DeepCellConfig()

# DeepCell Mesmer parameters
config.compartment_mode = "both"  # 'nuclear', 'whole-cell', or 'both'
config.primary_output_mask = "cell"  # 'cell' or 'nuclear' (if mode is 'both')
config.merge_membrane_markers = True # Max projection of all markers

# Output options
config.save_geojson = True   # Save cell boundaries as GeoJSON
config.save_npy = False      # Save segmentation mask as NPY file
config.save_png = False      # Save segmentation visualization as PNG

# Optional tissue masking
config.use_input_geojson = False  # Set to True to use tissue annotations
config.input_geojson_path = "../data/json/tissue_annotations.geojson"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"File pattern: {FILE_PATTERN}")
print(f"DAPI channel: {DAPI_CHANNEL}")
print(f"Segmentation markers: {len(SEGMENTATION_MARKERS)} markers")
print(f"DeepCell compartment mode: {config.compartment_mode}")
print(f"Use tissue masking: {config.use_input_geojson}")

## Check Available Images

Let's first check what images are available for processing.

In [ ]:
import glob
import os

# Find all matching images
search_pattern = os.path.join(INPUT_DIR, FILE_PATTERN)
image_files = glob.glob(search_pattern)

print(f"Found {len(image_files)} images matching pattern: {search_pattern}")

if image_files:
    print("\n" + "="*80)
    print("AVAILABLE IMAGES FOR PROCESSING")
    print("="*80)
    
    # Create a dictionary to store image information
    image_info = {}
    
    for i, img_path in enumerate(image_files, 1):
        filename = os.path.basename(img_path)
        file_size = os.path.getsize(img_path) / (1024**3)  # Size in GB
        
        # Check if outputs already exist
        existing_outputs = check_segmentation_outputs_exist(img_path, OUTPUT_DIR, config)
        
        status = "✓ Processed" if existing_outputs['all_exist'] else "○ Not processed"
        
        print(f"  {i:2d}. {filename}")
        print(f"      Size: {file_size:.2f} GB | Status: {status}")
        
        image_info[i] = {
            'path': img_path,
            'filename': filename,
            'processed': existing_outputs['all_exist']
        }
        
else:
    print(f"\n⚠ No images found! Please check:")
    print(f"  - Input directory exists: {os.path.exists(INPUT_DIR)}")
    print(f"  - File pattern is correct: {FILE_PATTERN}")
    image_info = {}

## Execute Segmentation

This cell runs the DeepCell segmentation pipeline on the selected images.

### A Note on Dependencies

This notebook requires the `deepcell` library. If you encounter an error in the next cell stating that `DeepCell is not available`, please install the required packages by running the command below in your terminal from the `SPACEMAP_V4` project root, then **restart the Jupyter kernel** and try again.

```bash
pip install -r ../requirements.txt
```

In [ ]:
# ============================================================================
# EXECUTE SEGMENTATION PROCESSING
# ============================================================================

# Processing mode: 'single' or 'batch'
PROCESS_MODE = "batch"
SINGLE_IMAGE_INDEX = 1  # Used if PROCESS_MODE is 'single'
OVERWRITE_EXISTING = False # Set to True to re-process existing files

if not image_files:
    print(" No images to process.")
else:
    images_to_process = []
    if PROCESS_MODE == "single":
        if SINGLE_IMAGE_INDEX in image_info:
            images_to_process.append(image_info[SINGLE_IMAGE_INDEX])
    else: # batch
        images_to_process = list(image_info.values())
        
    print(f"\n Starting DeepCell segmentation of {len(images_to_process)} image(s)...")
    print("="*60)
    
    processing_results = {}
    
    for i, image_data in enumerate(images_to_process, 1):
        if not image_data['processed'] or OVERWRITE_EXISTING:
            print(f"\n--- Processing {i}/{len(images_to_process)}: {image_data['filename']} ---")
            try:
                results = segment_image_deepcell(
                    ome_tiff_path=image_data['path'],
                    output_dir=OUTPUT_DIR,
                    config=config,
                    markers=SEGMENTATION_MARKERS,
                    dapi_channel=DAPI_CHANNEL,
                    overwrite=OVERWRITE_EXISTING
                )
                processing_results[image_data['filename']] = results
                print(f" SUCCESS: Processing complete for {image_data['filename']}")
            except Exception as e:
                print(f" FAILED: {str(e)}")
                import traceback
                traceback.print_exc()
        else:
            print(f"Skipping {image_data['filename']} - already processed.")

    # Store the last result for visualization
    if processing_results:
        last_successful = next((res for res in processing_results.values() if 'error' not in res), None)
        if last_successful:
            globals()['results'] = last_successful
            print("\n Results from last processed image stored for visualization.")

## Visualize Segmentation Results

Let's create some visualizations to check the quality of segmentation.

In [ ]:
if 'results' in locals() and 'features_df' in results:
    features_df = results['features_df']
    masks = results['masks']
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. Cell area distribution
    axes[0].hist(features_df['area'], bins=50, alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Area (pixels)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title(f'Distribution of {config.primary_output_mask.capitalize()} Areas')
    axes[0].grid(True, alpha=0.3)
    
    # 2. Segmentation mask overview
    display_mask = masks[::4, ::4] # Downsample for display
    axes[1].imshow(display_mask, cmap='tab20', interpolation='nearest')
    axes[1].set_title(f'Segmentation Mask ({len(np.unique(masks))-1} objects)')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Segmentation Summary ===")
    print(f"Total objects detected: {len(features_df)}")
    print(f"Mean area: {features_df['area'].mean():.1f} ± {features_df['area'].std():.1f} pixels")
    
    print(f"\nNext steps in SPACEMAP pipeline:")
    print(f"  - Step4: Cell type annotation")
    print(f"  - Step5: Batch correction (if needed)")
    print(f"  - Step6A-D: Phenotyping analysis")
    
else:
    print("No segmentation results available to visualize.")